# Валидация

Главная цель данного этапа - понять, как необходимо разделить данные на обучающую и валидационную выборки, которая имитирует работу реальной рекомендательной системы.

Для каждой сессии часть более ранних событий будет использовать как уже известная история сессий пользователей, а последующие события - как целевые переменные, которая рекомендательная система должна предсказать. 

В данной задаче нельзя случайно перемешать строки через train_test_split, так как это может привести к утечке данных (таким образом дадим модели информацию из будущего)

In [1]:
import pandas as pd
from pathlib import Path 

data_path = Path("../data/otto-recsys-train.jsonl")

df = pd.read_json(
    data_path,
    lines=True,
    nrows=10_000
)

df.head()

,session,events
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ..."


## Разделение сессии

Каждую сессии разделим таким образом:

- первые 75% событий - известная история ('history');
- последние 25% событий - будущие события ('future'), которые необходимо предсказать.

В случае, если сессия является слишком короткой, то в 'future' оставляем как минимум одно событие

In [3]:
events = df.loc[0, 'events']

future_size = max(1, int(len(events) * 0.25))

history = events[:-future_size]
future = events[-future_size:]

print('Всего событий:', len(events))
print('History:', len(history))
print('Future:', len(future))

Всего событий: 276
History: 207
Future: 69


## Разделение всех сессий

Перенесем эту логику на все сессии выборки.

In [6]:
history_arr = []
future_arr = []

for events in df['events']:
    future_size = max(1, int(len(events) * 0.25))

    history = events[:-future_size]
    future = events[-future_size:]

    history_arr.append(history)
    future_arr.append(future)

df['history'] = history_arr
df['future'] = future_arr

df[['session', 'history', 'future']].head()

,session,history,future
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':...","[{'aid': 362233, 'ts': 1661551907312, 'type': ..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ...","[{'aid': 959544, 'ts': 1659991199405, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ...","[{'aid': 295985, 'ts': 1661713601647, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':...","[{'aid': 88711, 'ts': 1659998456613, 'type': '..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ...","[{'aid': 758750, 'ts': 1661504510200, 'type': ..."


## Проверка разбиения

Стоит проверить корректность разбиения сессий. Делать это будем по следующим критериям:

- 'history' и 'future' не должны быть пустыми;
- общая сумма 'history' и 'future' должна совпадать с количеством всех событий текущей сессии;
- события должы идти в хронологическом порядке и не нарушать логику (последнее событие 'history' не должно быть позже первого события 'future').

In [11]:
empty_rows = 0 
wrong_size = 0
wrong_timeline = 0

for events, history, future in zip(
    df['events'],
    df['history'],
    df['future']
):
    if len(history) == 0:
        empty_rows += 1

    if len(future) == 0:
        empty_rows += 1

    if len(history) + len(future) != len(events):
        wrong_size += 1

    if history[-1]['ts'] > future[0]['ts']:
        wrong_timeline += 1

print('Пустых частей:', empty_rows)
print('Ошибок в размере:', wrong_size)
print('Ошибок в хронологии:', wrong_timeline)

Пустых частей: 0
Ошибок в размере: 0
Ошибок в хронологии: 0


Разбиение сессий оказалось корректным.

## Целевые события

В выборке присутствуют три типа взаимодействия с товарами:

- `clicks` — просмотр товара;
- `carts` — добавление товара в корзину;
- `orders` — заказ товара.

Эти типы взаимодействия имеют разную ценность. Заказ являестся более сильным сигналом, чем добавление товара в корзину, а добавление в корзину - более сильным, чем просмотр товара. 

Поэтому качество рекомендаций будем анализировать отдельно для этих трех событий.

## Формирование целевых товаров

Разделим товары из будущей части каждой сессии по типу взаимодействия.

Для каждой сессии сформируем три набора целевых товаров:

- `future_clicks` — товары, по которым произошёл клик;
- `future_carts` — товары, добавленные в корзину;
- `future_orders` — заказанные товары.

Один товар будем учитывать один раз внутри каждого типа события, даже если пользователь взаимодействовал с ним несколько раз, посколько нам важно понять, смогла ли система порекомендовать нужный товар.

Повторные действия с товаром в 'history' сохраняем, так как они показывают силу интереса пользователя.

In [13]:
clicks_arr = []
carts_arr = []
orders_arr = []

for future in df['future']:
    clicks = set()
    carts = set()
    orders = set()

    for event in future:
        if event['type'] == 'clicks':
            clicks.add(event['aid'])

        elif event['type'] == 'carts':
            carts.add(event['aid'])

        elif event['type'] == 'orders':
            orders.add(event['aid'])

    clicks_arr.append(clicks)
    carts_arr.append(carts)
    orders_arr.append(orders)

df['future_clicks'] = clicks_arr
df['future_carts'] = carts_arr
df['future_orders'] = orders_arr

In [15]:
df[['session', 'future_clicks', 'future_carts', 'future_orders']].head()

,session,future_clicks,future_carts,future_orders
0,0,"{1319939, 976134, 883849, 315914, 168206, 1760...","{315914, 543308, 1549618, 1199474, 442293}","{1199474, 543308}"
1,1,"{1842593, 207905, 376932, 1464360, 497868, 173...",{},{}
2,2,"{485582, 1605711, 808782, 295985, 553269, 5550...",{},{}
3,3,"{618373, 88711, 1152906, 1536010, 1175830, 672...","{925352, 1638009, 1261998}","{54857, 1018433}"
4,4,"{678521, 479396, 1081407}",{758750},{}


## Распределение целевых событий

Посмотрим, в какой доле сессий в 'future' есть клики, добавления в корзину и заказы.

Это поможет понять, насколько часто встречается каждый тип целевого события.

In [16]:
click_sessions = (df['future_clicks'].apply(len) > 0).mean()
cart_sessions = (df['future_carts'].apply(len) > 0).mean()
order_sessions = (df['future_orders'].apply(len) > 0).mean()

print(f'Сессии с clicks: {click_sessions:.2%}')
print(f'Сессии с carts: {cart_sessions:.2%}')
print(f'Сессии с orders: {order_sessions:.2%}')

Сессии с clicks: 98.62%
Сессии с carts: 32.72%
Сессии с orders: 15.65%


### Вывод

Клики присутствуют почти во всех сессиях (98.62%), добавления в корзину — примерно в трети сессий (32.72%), а заказы — только в 15.65%.

Поэтому качество рекомендаций важно оценивать отдельно для каждого типа события: одна общая метрика может скрыть плохое качество на более редких `carts` и `orders`.

Теперь настало время выбрать метрики, которые будут оценивать качество рекомендаций.

## Метрики качества рекомендаций

Для оценки топ-20 рекомендаций будем использовать три метрики:

- `Recall@20` — какую долю релевантных товаров из 'future' система смогла найти;
- `Precision@20` — какая доля рекомендованных товаров оказалась релевантной;
- `NDCG@20` — учитывает не только попадание релевантного товара в рекомендации, но и его позицию в списке (товара на первых местах дается больший приоритет).

Метрики будем считать отдельно для `clicks`, `carts` и `orders`.

In [23]:
def recall_at_k(recommend, relevant_items, k=20):
    recommend = recommend[:k]

    hits = 0

    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / len(relevant_items)



recommend = [10, 20, 30, 40, 50]
relevant_items = {20, 30, 70, 80}

recall_at_k(recommend, relevant_items, k=5)

0.5

In [26]:
def precision_at_k(recommend, relevant_items, k=20):
    recommend = recommend[:k]

    hits = 0

    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / k



recommend = [10, 20, 30, 40, 50]
relevant_items = {20, 30, 70, 80}

precision_at_k(recommend, relevant_items, k=5)

0.4

Теперь стоит рассмотреть граничные случаи:

- В future нет нужно события;
- Рекомендаций меньше k.

Исправим код, чтобы он учитывал эти случаи.

In [28]:
def recall_at_k(recommend, relevant_items, k=20):
    if len(relevant_items) == 0:
        return None

    recommend = recommend[:k]

    hits = 0

    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / len(relevant_items)

In [29]:
def precision_at_k(recommend, relevant_items, k=20):
    recommend = recommend[:k]

    hits = 0

    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / k

### NDCG@K

`NDCG@K` учитывает не только попадание релевантного товара в топ-K, но и его позицию.

Чем выше находится релевантный товар в списке рекомендаций, тем больший вклад он вносит в метрику.

$$
\frac{1}{\log_2(\text{position} + 1)}
$$

In [33]:
import math 

def ndcg_at_k(recommend, relevant_items, k=20):
    if len(relevant_items) == 0:
        return None 

    recommned = recommend[:k]
    dcg = 0

    for pos, item in enumerate(recommend, start=1):
        if item in relevant_items:
            dcg += 1 / math.log2(pos + 1)

    ideal_hits = min(len(relevant_items), k)

    idcg = 0
    for pos in range(1, ideal_hits + 1):
        idcg += 1 / math.log2(pos + 1)

    return dcg / idcg




recommend = [10, 20, 30, 40, 50]
relevant_items = {20, 30, 70, 80}
k = 5

print('Рекомендации:', recommend)
print('Релевантные товары:', relevant_items)
print('Количество рекомендаций:', k)

print('\nРелевантные товары в рекомендациях:')

for pos, item in enumerate(recommend[:k], start=1):
    if item in relevant_items:
        print(f'Позиция {pos}: товар {item} — релевантный')
    else:
        print(f'Позиция {pos}: товар {item} — не релевантный')

result = ndcg_at_k(recommend, relevant_items, k)

print(f'\nNDCG@{k}: {result:.3f}')

Рекомендации: [10, 20, 30, 40, 50]
Релевантные товары: {80, 20, 70, 30}
Количество рекомендаций: 5

Релевантные товары в рекомендациях:
Позиция 1: товар 10 — не релевантный
Позиция 2: товар 20 — релевантный
Позиция 3: товар 30 — релевантный
Позиция 4: товар 40 — не релевантный
Позиция 5: товар 50 — не релевантный

NDCG@5: 0.441


## Итоговая схема оценки

Качество рекомендательной системы будем оценивать с помощью:

- **Recall@20** — какую долю релевантных товаров система смогла найти;
- **Precision@20** — какая доля рекомендованных товаров оказалась релевантной;
- **NDCG@20** — насколько высоко система расположила релевантные товары.

Метрики будем считать отдельно для `clicks`, `carts` и `orders`, поскольку эти действия имеют разную значимость.

Сессии, в которых в 'future' отсутствует соответствующий тип события, не будем учитывать при расчёте Recall и NDCG для этого типа.